# word2vec — Efficient Estimation of Word Representations in Vector Space (2013)

_Papers · Sequence & NLP_

**Learn a dense vector for each word by predicting its neighbors — cheaply enough to train on billions of words.**

---

This notebook is a practice scaffold for the **word2vec — Efficient Estimation of Word Representations in Vector Space (2013)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(0)

class MySkipGram:
    """Skip-gram with full softmax — raw tensors, autograd on.
       V_in[w]  = input/center vector  e_c  (kept as the word's final embedding)
       V_out[w] = output/context vector theta_w (used only to score targets)."""
    def __init__(self, vocab, dim, scale=0.1):
        self.V_in  = (torch.randn(vocab, dim) * scale).requires_grad_(True)
        self.V_out = (torch.randn(vocab, dim) * scale).requires_grad_(True)

    def loss(self, center, context):
        e_c    = self.V_in[center]                       # (B, D) center vectors
        scores = e_c @ self.V_out.t()                    # (B, V) theta_j . e_c for every word j
        logZ   = torch.logsumexp(scores, dim=1)          # (B,) log softmax denominator (stable)
        gold   = scores[torch.arange(len(center)), context]   # (B,) numerator score of true target
        return (logZ - gold).mean()                      # mean negative log-likelihood

# ---- THE ORACLE: my hand-rolled loss must equal F.cross_entropy on the same scores ----
V, D = 6, 4
m = MySkipGram(V, D)
center  = torch.tensor([0, 2, 4])
context = torch.tensor([1, 3, 5])
mine = m.loss(center, context)
with torch.no_grad():
    scores = m.V_in[center] @ m.V_out.t()
ref = F.cross_entropy(scores, context)                  # PyTorch's softmax + NLL in one call
print("my skip-gram loss :", round(mine.item(), 6))     # 1.796847
print("F.cross_entropy   :", round(ref.item(), 6))      # 1.796847
print("allclose          :", torch.allclose(mine, ref, atol=1e-6))   # True

# ---- recompute the WORKED EXAMPLE: e_c=[1,2], thetas=[[1,0],[0.5,1],[-1,1]], target=1 ----
e_c    = torch.tensor([1.0, 2.0])
thetas = torch.tensor([[1.0, 0.0], [0.5, 1.0], [-1.0, 1.0]])
sc     = thetas @ e_c
probs  = torch.softmax(sc, dim=0)
print("worked scores     :", sc.tolist())               # [1.0, 2.5, 1.0]
print("worked softmax    :", [round(p, 6) for p in probs.tolist()])  # [0.154281, 0.691438, 0.154281]
print("worked loss(t=1)  :", round(-torch.log(probs[1]).item(), 6))  # 0.368981

# ---- train on a tiny two-topic corpus, then nearest neighbors ----
torch.manual_seed(1)
corpus = ("king queen man woman king queen prince princess man woman boy girl "
          "king prince man boy queen princess woman girl cat dog cat dog kitten puppy "
          "cat kitten dog puppy king queen prince princess").split()
vocab = sorted(set(corpus)); stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
pairs = []                                               # window = 2 on each side
for i, w in enumerate(corpus):
    for j in range(max(0, i - 2), min(len(corpus), i + 3)):
        if j != i:                                       # a word is not its own context
            pairs.append((stoi[w], stoi[corpus[j]]))
ctr = torch.tensor([a for a, _ in pairs]); ctx = torch.tensor([b for _, b in pairs])

sg  = MySkipGram(V, 8)
opt = torch.optim.Adam([sg.V_in, sg.V_out], lr=0.05)
for epoch in range(300):
    opt.zero_grad()
    L = sg.loss(ctr, ctx)
    L.backward(); opt.step()
print("final train loss  :", round(L.item(), 4))         # ~1.73

E  = sg.V_in.detach()                                     # keep the INPUT table as the embeddings
En = E / E.norm(dim=1, keepdim=True)
sim = En @ En.t()                                         # cosine similarity (rows normalized)
def neighbors(w, k=3):
    i = stoi[w]; s = sim[i].clone(); s[i] = -9            # exclude the word itself
    return [vocab[t] for t in s.topk(k).indices.tolist()]
for w in ["king", "cat", "man"]:
    print(f"nearest to {w:6s}:", neighbors(w))
# e.g. -> cat: ['dog','kitten','puppy']  king: ['princess','queen','boy']

## Visualize the data & results

_Train skip-gram on a tiny corpus that mixes royalty/people words and animal words — do the learned embeddings cluster by topic without ever being told the topics?_

In [ ]:
import torch
torch.manual_seed(1)
corpus = ("king queen man woman king queen prince princess man woman boy girl "
          "king prince man boy queen princess woman girl cat dog cat dog kitten puppy "
          "cat kitten dog puppy king queen prince princess man woman boy girl "
          "cat dog kitten puppy").split()
vocab = sorted(set(corpus)); stoi = {w: i for i, w in enumerate(vocab)}; V = len(vocab)
pairs = []
for i, w in enumerate(corpus):
    for j in range(max(0, i - 2), min(len(corpus), i + 3)):
        if j != i:
            pairs.append((stoi[w], stoi[corpus[j]]))
ctr = torch.tensor([a for a, _ in pairs]); ctx = torch.tensor([b for _, b in pairs])

Vin  = (torch.randn(V, 8) * 0.1).requires_grad_(True)
Vout = (torch.randn(V, 8) * 0.1).requires_grad_(True)
opt  = torch.optim.Adam([Vin, Vout], lr=0.05)
for epoch in range(400):
    opt.zero_grad()
    sc = Vin[ctr] @ Vout.t()
    L  = (torch.logsumexp(sc, 1) - sc[torch.arange(len(ctr)), ctx]).mean()
    L.backward(); opt.step()

E   = Vin.detach()
Ec  = E - E.mean(0)                                   # center, then PCA via SVD
U, S, Vt = torch.linalg.svd(Ec, full_matrices=False)
P   = (Ec @ Vt[:2].t())
P   = P / P.abs().max() * 2.0                          # scale to a readable range
for w in vocab:
    x, y = P[stoi[w]].tolist()
    print(f"{w:9s} {x:6.3f} {y:6.3f}")                 # the labeled coordinates plotted above

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute the skip-gram softmax for center $e_c=[2,1]$ with output vectors $\theta_0=[1,0]$, $\theta_1=[0,2]$, $\theta_2=[1,1]$, and give the loss if the true target is word $2$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scores: $\theta_0^\top e_c=2$, $\theta_1^\top e_c=2$, $\theta_2^\top e_c=3$. — _Dot product = the raw score per word._
- Exponentiate: $[e^2,e^2,e^3]=[7.389,7.389,20.086]$; sum $=34.864$. — _Numerator candidates and the softmax denominator._
- Probabilities: $[0.2119,0.2119,0.5762]$ (sum 1). — _Divide each by the total._
- Loss for target 2: $-\ln(0.5762)=0.5514$. — _Negative log-likelihood of the true context word._

**Answer:** Probabilities $\approx[0.212,0.212,0.576]$; loss $\approx 0.551$. Word 2 has the highest dot product with $e_c$, so it gets the most probability and the smallest loss.

</details>

**Problem 2.** Using Eq. 5, $Q=C(D+D\log_2 V)$, compare the per-example cost for $C=10,\ D=300$ at $V=1000$ vs $V=1{,}000{,}000$. Why does the model still scale?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\log_2 1000\approx 9.97$, so $Q=10(300+300\cdot9.97)\approx 10\cdot3290=32{,}900$. — _Plug in the small vocabulary._
- $\log_2 10^6\approx 19.93$, so $Q=10(300+300\cdot19.93)\approx 10\cdot6280=62{,}800$. — _A 1000x bigger vocabulary._
- Cost roughly doubles, not 1000x. — _The output term grows like $\log_2 V$, not $V$._

**Answer:** A thousand-fold larger vocabulary only about doubles the cost, because the hierarchical-softmax output term is $D\log_2 V$. A plain softmax would have a $D\cdot V$ term and blow up linearly with $V$ — that is the whole point of Eq. 5.

</details>

**Problem 3.** Ablation: in the CODE, replace the two separate tables (input $e$ and output $\theta$) with a single shared table used for both center and context. What changes, and what should stay the same?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set V_out = V_in (one tensor) and retrain on the same corpus. — _Ties each word's input and output vector together._
- Re-check the torch.allclose oracle. — _It only tests the loss formula, not the table structure, so it should still pass._
- Re-read the nearest neighbors / scatter. — _See whether the topic clusters survive._

**Answer:** The allclose check still passes — it verifies the softmax-cross-entropy loss, which does not care whether the tables are shared. The clustering usually still appears on this toy data, but a shared table is a different (more constrained) model than the paper's skip-gram; with one table a word's dot product with itself is its squared norm, which can distort similarities. Keeping the tables separate matches Section 3.2.

</details>